<a href="https://colab.research.google.com/github/rifahnanjiba02-arch/selective-decoder-routing/blob/main/notebooks/00_dataset_and_feasibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Selective Decoder Routing — Dataset and Feasibility Analysis

**Week 1 | Incremental Colab notebook**

**Dataset:** [Bauxitiego/surface-code-syndromes](https://huggingface.co/datasets/Bauxitiego/surface-code-syndromes) — a reformatted version of experimental surface-code data from Google Quantum AI.

**Research question:** How often could choosing correlated matching instead of PyMatching correct an otherwise incorrect logical prediction?

Each row is an experimental shot containing a syndrome, the true logical `label`, and **stored predictions** from `pymatching` and `correlated_matching`. We do not run either decoder during this feasibility study.

**Scope:** Use the published **training** split for Week 1 development. The published validation split is reserved for model/threshold selection; the **test split must remain untouched** until final evaluation. This notebook does not train ML or QML models. Ground-truth labels and stored decoder predictions may be used to create routing targets and evaluate decoders, but must **never** become router input features.

## 1. Minimal setup

Download only the Python modules, configuration, and tests needed for this notebook from **one resolved GitHub commit**. No repository clone, model download, QML package, or `pip install -e .` is needed for this small subset. The original implementation remains in `src/router/`; do not copy its calculations into notebook cells.

For final reproducibility, replace `SOURCE_REF = "main"` with the resolved commit SHA printed below.

In [1]:
%pip install -q datasets numpy pandas PyYAML

In [7]:
import json
import os
import sys
from pathlib import Path
from urllib.request import urlopen

REPO = "rifahnanjiba02-arch/selective-decoder-routing"
SOURCE_REF = "main"  # Use the printed commit SHA for a frozen final run.
ROOT = Path("/content/selective_decoder_week1")

with urlopen(f"https://api.github.com/repos/{REPO}/commits/{SOURCE_REF}") as response:
    repo_commit = json.load(response)["sha"]

required_files = [
    "configs/data.yaml",
    "src/router/__init__.py",
    "src/router/data.py",
    "src/router/decoders.py",
    "src/router/labels.py",
    "src/router/routing.py",
    "tests/test_data.py",
    "tests/test_routing.py",
]

for relative_path in required_files:
    destination = ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    url = f"https://raw.githubusercontent.com/{REPO}/{repo_commit}/{relative_path}"
    with urlopen(url) as response:
        destination.write_bytes(response.read())

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print(f"Loaded {len(required_files)} project files at commit: {repo_commit}")

Loaded 8 project files at commit: f05da60e7c496bb014b427de23417713af538dc3


## 2. Run settings

The team configuration currently leaves `dataset_revision: null`. To prevent dataset changes from silently altering this notebook's sample, use the **verified Hugging Face dataset commit** below in a *temporary local copy* of the config. The repository's original config is not modified.

In [8]:
import yaml

DATASET_REVISION = "9c51b27ffc51ed7ff3ef3a72167ebf4862d1f2f3"

with open(ROOT / "configs/data.yaml", encoding="utf-8") as file:
    config = yaml.safe_load(file)

assert config["dataset_name"] == "Bauxitiego/surface-code-syndromes"
assert config.get("dataset_revision") in (None, DATASET_REVISION)

pinned_config = dict(config)
pinned_config["dataset_revision"] = DATASET_REVISION
config_path = ROOT / "configs/data_colab_pinned.yaml"
config_path.write_text(yaml.safe_dump(pinned_config, sort_keys=False), encoding="utf-8")

run_settings = {
    "repository_commit": repo_commit,
    "dataset": pinned_config["dataset_name"],
    "dataset_revision": DATASET_REVISION,
    "split": "train",
    "sample_size": pinned_config["development"]["train_size"],
    "sampling_seed": pinned_config["sampling_seed"],
    "published_split_seed": pinned_config["source_split_seed"],
}
for name, value in run_settings.items():
    print(f"{name}: {value}")

repository_commit: f05da60e7c496bb014b427de23417713af538dc3
dataset: Bauxitiego/surface-code-syndromes
dataset_revision: 9c51b27ffc51ed7ff3ef3a72167ebf4862d1f2f3
split: train
sample_size: 50000
sampling_seed: 42
published_split_seed: 20260806


## 3. Existing offline unit tests

Run the teammate-written tests before loading real data. These tests check function behavior on small inputs; they **do not** establish results on the real experimental dataset.

In [10]:
import os
import sys
import subprocess
from pathlib import Path

ROOT = Path("/content/selective_decoder_week1")

# Check that the setup downloaded the required files
assert (ROOT / "tests").is_dir(), "Run the setup cell first."
assert (ROOT / "src/router").is_dir(), "Router files are missing."

env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT / "src")

subprocess.run(
    [sys.executable, "-m", "unittest", "discover",
     "-s", "tests", "-v"],
    cwd=ROOT,
    env=env,
    check=True
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], returncode=0)

## 4. Load the real training data

Call the **existing** `load_surface_code_dataset()` and `sample_split()` functions. The loader requests the `train` split and the notebook selects rows only from `train`. This run's Hugging Face builder also downloaded/prepared validation and test files while loading; no validation or test rows were selected for analysis. The full training split is prepared before the 50,000-shot sample is selected, so this is not a streaming sample.

Published dataset-card split sizes: train **4,550,000**, validation **650,000**, test **1,300,000**. This notebook directly confirms the train count; the validation and test counts are taken from the card.

In [11]:
from router.data import load_surface_code_dataset, sample_split

splits, loaded_config = load_surface_code_dataset(
    config_path=str(config_path), split_name="train"
)

sample = sample_split(
    splits,
    split_name="train",
    sample_size=loaded_config["development"]["train_size"],
    seed=loaded_config["sampling_seed"],
)

assert len(sample) == run_settings["sample_size"]
print("Full training split:", len(splits["train"]))
print("Development sample:", len(sample))
print("Columns:", sample.column_names)

README.md:   0%|          | 0.00/6.90k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B / 69.9MB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 10.0MB            

validation.parquet: downloading bytes:           |  0.00B            

...

test.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Full training split: 4550000
Development sample: 50000
Columns: ['experiment', 'basis', 'distance', 'rounds', 'time_steps', 'width', 'round_widths', 'syndrome', 'label', 'pymatching', 'correlated_matching', 'tensor_network']


In [12]:
# Check that the same seed reproduces the sample's stored labels/predictions.
repeat = sample_split(
    splits,
    split_name="train",
    sample_size=run_settings["sample_size"],
    seed=run_settings["sampling_seed"],
)
for column in ("label", "pymatching", "correlated_matching"):
    assert list(sample[column]) == list(repeat[column])
print("Repeated sampling agrees on all stored label/prediction columns.")

Repeated sampling agrees on all stored label/prediction columns.


In [13]:
# Compact preview: do not print full syndrome arrays.
from IPython.display import display

preview_columns = [
    "experiment", "basis", "distance", "rounds",
    "label", "pymatching", "correlated_matching",
]
display(sample.select(range(5)).select_columns(preview_columns).to_pandas())

,experiment,basis,distance,rounds,label,pymatching,correlated_matching
0,surface_code_bZ_d3_r13_center_5_7,Z,3,13,1,1,1
1,surface_code_bZ_d3_r03_center_5_3,Z,3,3,0,0,0
2,surface_code_bX_d3_r01_center_3_5,X,3,1,0,0,0
3,surface_code_bZ_d5_r13_center_5_5,Z,5,13,0,0,0
4,surface_code_bZ_d5_r03_center_5_5,Z,5,3,0,0,0


## 5. Remaining data inspection

### Observed value sets

### Missing counts and data types

### Syndrome / `round_widths` shape checks

### Unresolved schema notes

## 6. Decoder outcome results

### Four-count table

### Five-rate table

## 7. Routing target

## 8. Checks and interpretation

## 9. Compact findings note — `results/feasibility/`